# Embedding vector math via `skainet-transformers` (Spring-AI-shaped API)

Self-contained Kotlin notebook: pulls `skainet-transformers` artifacts from Maven Central via `@file:DependsOn`. Demonstrates `king − man + woman ≈ queen` against the Spring-AI-shaped `EmbeddingModel` SPI in `sk.ainet.llm.api`.

**Versions are BOM-aligned.** Gradle/Maven users get this for free with the `sk.ainet.transformers:skainet-transformers-bom:0.23.5` platform; in a Kotlin notebook the BOM itself can't be used in `@file:DependsOn` (BOMs ship POM-only — no JAR for the resolver to download), so each artifact is pinned to the version the BOM resolves it to:

| Group | Version |
|-------|---------|
| `sk.ainet.transformers:*` | `0.23.5` |
| `sk.ainet.core:*` (re-exported via `sk.ainet:skainet-bom:0.23.1`) | `0.23.1` |

The Spring-AI parallel:

| Spring AI | `sk.ainet.llm.api` |
|-----------|--------------------|
| `EmbeddingModel.call(EmbeddingRequest)` | identical |
| `EmbeddingResponse(embeddings, metadata)` | `EmbeddingResponse(embeddings, usage, modelId)` |
| `Embedding(output, index)` | `Embedding(index, vector)` |

**Caveat:** LEAF (`mdbr-leaf-mt`) is a *sentence-transformers* BERT, not word2vec. The analogy ranks the right answer near the top but with a smaller gap than static word embeddings.

## 1. Resolve dependencies and import

In [1]:
@file:DependsOn("sk.ainet.transformers:skainet-transformers-providers:0.23.5")
@file:DependsOn("sk.ainet.transformers:skainet-transformers-inference-bert:0.23.5")
@file:DependsOn("sk.ainet.core:skainet-lang-core:0.23.1")
@file:DependsOn("sk.ainet.core:skainet-io-core:0.23.1")
@file:DependsOn("sk.ainet.core:skainet-io-safetensors:0.23.1")
@file:DependsOn("sk.ainet.core:skainet-backend-cpu:0.23.1")
@file:DependsOn("org.jetbrains.kotlinx:kotlinx-coroutines-core:1.10.2")

import sk.ainet.context.DirectCpuExecutionContext
import sk.ainet.io.JvmRandomAccessSource
import sk.ainet.io.safetensors.SafeTensorsParametersLoader
// Wildcard import brings the tensor operator extensions (plus / minus / times /
// div / cosineDistance) AND the types (Tensor, Shape) into scope. The operators
// are top-level extension functions in TensorExtensions.kt — without the
// wildcard the kernel resolves `tensor1 - tensor2` against BigDecimal /
// Iterable / etc. and fails.
import sk.ainet.lang.tensor.*
import sk.ainet.lang.types.FP32
import sk.ainet.llm.api.EmbeddingModel
import sk.ainet.llm.api.EmbeddingRequest
import sk.ainet.llm.api.EmbeddingResponse
import sk.ainet.llm.providers.SkaiNetEmbeddingModel
import sk.ainet.models.bert.BertRuntime
import sk.ainet.models.bert.HuggingFaceTokenizer
import sk.ainet.models.bert.MDBR_LEAF_IR_CONFIG
import sk.ainet.models.bert.loadBertWeights
import kotlinx.coroutines.runBlocking
import java.nio.file.Path
import kotlin.io.path.Path
import kotlin.io.path.exists
import kotlin.io.path.readText

## 2. Construct the model behind the SPI

Until upstream ships `BertEmbeddingModel.load(...)` (described in the skainet-transformers PRD), construction needs the BERT-specific assembly: tokenizer, weights loader, runtime, and the `SkaiNetEmbeddingModel` adapter. Once `BertEmbeddingModel.load(...)` lands, this cell collapses to one line.

Adjust `modelDir` to point at your local snapshot of [MongoDB/mdbr-leaf-ir](https://huggingface.co/MongoDB/mdbr-leaf-ir) — the directory must contain `vocab.txt` and `model.safetensors`.

In [21]:
fun loadLeafModel(modelDir: Path, ctx: DirectCpuExecutionContext): EmbeddingModel {
    val tokenizer = HuggingFaceTokenizer.fromVocabTxt(modelDir.resolve("vocab.txt").readText())

    val baseWeights = sequenceOf("model.safetensors", "pytorch_model.safetensors")
        .map { modelDir.resolve(it) }
        .firstOrNull { it.exists() }
        ?: error("No safetensors file in $modelDir")

    // Optional sentence-transformers dense projection layer. mdbr-leaf-mt
    // ships one (output 768d); mdbr-leaf-ir does not (output stays at 384d).
    // MDBR_LEAF_IR_CONFIG hardcodes projectionDim = 768, so we override to
    // null when the dense layer isn't present — otherwise the SPI's reported
    // dimensions disagree with the actual embedding size.
    val denseWeights = modelDir.resolve("2_Dense/model.safetensors").takeIf { it.exists() }
    val config = if (denseWeights != null) MDBR_LEAF_IR_CONFIG else MDBR_LEAF_IR_CONFIG.copy(projectionDim = null)

    val loaders = listOfNotNull(baseWeights, denseWeights).map { file ->
        SafeTensorsParametersLoader(
            sourceProvider = { JvmRandomAccessSource.open(file.toString()) },
            onProgress = { _, _, _ -> },
        )
    }

    val weights = runBlocking { loadBertWeights(loaders, ctx, FP32::class, config) }
    val runtime = BertRuntime(ctx, weights, FP32::class)
    return SkaiNetEmbeddingModel(
        runtime = runtime,
        tokenizer = tokenizer,
        dimensions = config.projectionDim ?: config.hiddenSize,
        modelId = modelDir.fileName.toString(),
    )
}

val ctx = DirectCpuExecutionContext()
val modelDir = Path("${System.getProperty("user.home")}/.deliverance/MongoDB_mdbr-leaf-ir")
val model: EmbeddingModel = loadLeafModel(modelDir, ctx)

println("dimensions = ${model.dimensions}")
println("dense layer = ${if (modelDir.resolve("2_Dense/model.safetensors").exists()) "yes (mdbr-leaf-mt)" else "no (mdbr-leaf-ir)"}")

[SKaiNET] Using standard CPU operations (Vector API not available)
dimensions = 384
dense layer = no (mdbr-leaf-ir)


## 3. Spring-AI flow: `call(EmbeddingRequest) → EmbeddingResponse`

One batched call instead of N round-trips. Indices in the response correspond to input order; we sort defensively. `Usage` reports prompt-token count for the whole batch.

In [31]:
val terms = listOf("king", "man", "woman","girl")
val candidates = listOf("queen", "princess", "duchess", "empress", "monarch", "wife", "lady", "king", "man", "woman")
val inputs = (terms + candidates).distinct()

val response: EmbeddingResponse = model.call(EmbeddingRequest(inputs))

println("embeddings:  ${response.embeddings.size}")
println("usage:       ${response.usage?.promptTokens} prompt tokens")
println("modelId:     ${response.modelId}")

val vectors: Map<String, FloatArray> = inputs.zip(
    response.embeddings.sortedBy { it.index }.map { it.vector }
).toMap()

embeddings:  11
usage:       33 prompt tokens
modelId:     MongoDB_mdbr-leaf-ir


## 4. Analogy via SKaiNET tensors

Operator overloads (`+ −`) and `cosineDistance` come from `sk.ainet.lang.tensor.TensorExtensions`. Lifting `FloatArray` → `Tensor<FP32, Float>` is one call against the same `ExecutionContext` the model uses, so subsequent ops dispatch through the Vector-API CPU backend.

In [32]:
fun tensor(text: String): Tensor<FP32, Float> =
    ctx.fromFloatArray(Shape(model.dimensions), FP32::class, vectors.getValue(text))

In [33]:
fun Tensor<FP32, Float>.scalar(): Float = data.copyToFloatArray()[0]


In [34]:
tensor("king")

sk.ainet.lang.tensor.operators.OpsBoundTensor@6cd16725

In [35]:
import sk.ainet.lang.tensor.*
val target = tensor("king") - tensor("man") + tensor("girl")


In [36]:
target


candidates.distinct()
    .map { it to (1f - target.cosineDistance(tensor(it)).scalar()) }
    .sortedByDescending { it.second }
    .forEach { (w, s) -> println("%-10s %.4f".format(w, s)) }

king       0.8438
princess   0.7956
queen      0.7702
lady       0.7389
monarch    0.7031
woman      0.6973
empress    0.6734
wife       0.6625
duchess    0.6566
man        0.5865


## Notes

- **Convenience shortcuts on `EmbeddingModel`:** `embed(text: String): FloatArray` and `embed(texts: List<String>): List<FloatArray>` are both implemented on top of `call(...)`. Use them when you don't need the `Usage` / `modelId` metadata.
- **Why root coords work in `@file:DependsOn`:** The skainet artifacts publish Gradle Module Metadata, and recent kotlin-jupyter versions (0.12+) honour it to pick the JVM variant automatically. Older kernels may need the explicit `-jvm` suffix (e.g. `skainet-transformers-providers-jvm:0.23.5`).
- **The optional dense projection layer (`2_Dense/model.safetensors`)** is skipped here. The full mdbr-leaf-mt snapshot ships one; the cell-2 loader matches LEAF's hardcoded `MDBR_LEAF_IR_CONFIG` so dimensions stay at 384 either way. To include the dense projection, add a second `SafeTensorsParametersLoader` for `modelDir.resolve("2_Dense/model.safetensors")` and pass both loaders to `loadBertWeights`.
- **Batch-rank with one matmul:** stack candidate vectors into a `[N, dim]` matrix, normalise rows, then `matmul` against the (normalised) query column to get all similarities in a single op — much faster than N individual `cosineDistance` calls when `N` grows.